# Student Data Cleaning / 学生数据清洗

整合了两个模板的清洗流程：
- 原教学模板（Data_Cleaning_Teaching_Bilingual.ipynb）
- 同学模板（sklearn 方法 + 箱线图 + 独热编码 + 数值缩放）


## 1. Import libraries / 导入库

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print('pandas:', pd.__version__)


pandas: 3.0.5


## 2. Read the data / 读取数据

In [2]:
DATA_FILE = Path('/workspace/.uploads/23091186-5ec3-4346-9498-dc542bc22971_student_data_messy.csv')

df = pd.read_csv(DATA_FILE)
print('Shape / 形状:', df.shape)
print('Duplicate rows / 重复行:', df.duplicated().sum())
print('Missing cells / 缺失单元格:', int(df.isna().sum().sum()))
df.head()


Shape / 形状: (31, 14)
Duplicate rows / 重复行: 1
Missing cells / 缺失单元格: 5


,Student_ID,Name,Age,Gender,Major,Attendance_pct,Homework,Presentation,Project,Enrollment_Date,Email,City,Height_cm,Weight_kg
0,1001,Ali Khan,20,Male,Computer Science,92.0,18.0,17.0,43,2026-02-15,ali@example.com,Wuhan,173,65.0
1,1002,Sara Li,19,F,computer science,88.0,19.0,18.0,45,15/02/2026,sara.li@example.com,Wuhan,165,52.0
2,1003,Zhang Wei,21,male,AI,95.0,20.0,19.0,48,2026/02/16,zhang.wei@example.com,Huangshi,178,70.0
3,1004,Fatima Noor,20,Female,Artificial Intelligence,NaN,16.0,17.0,40,"Feb 17, 2026",fatima.noor@example.com,Huangshi,160,50.0
4,1005,Chen Yu,twenty,M,CS,76.0,15.0,15.0,38,2026-02-18,chen.yu@example.com,wuhan,181,75.0


## 3. Inspect the data / 初步检查

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Student_ID       31 non-null     int64  
 1   Name             31 non-null     str    
 2   Age              31 non-null     str    
 3   Gender           30 non-null     str    
 4   Major            31 non-null     str    
 5   Attendance_pct   30 non-null     float64
 6   Homework         30 non-null     float64
 7   Presentation     30 non-null     float64
 8   Project          31 non-null     int64  
 9   Enrollment_Date  31 non-null     str    
 10  Email            31 non-null     str    
 11  City             31 non-null     str    
 12  Height_cm        31 non-null     int64  
 13  Weight_kg        30 non-null     float64
dtypes: float64(4), int64(3), str(7)
memory usage: 3.5 KB


In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Student_ID,31.0,NaN,NaN,NaN,1015.967742,9.038746,1001.0,1008.5,1016.0,1023.5,1030.0
Name,31,30,Amna Yousaf,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,31,7,20,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,30,9,Female,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Major,31,10,Data Science,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Attendance_pct,30.0,NaN,NaN,NaN,86.066667,7.864952,68.0,81.25,87.5,91.0,105.0
Homework,30.0,NaN,NaN,NaN,16.733333,4.540495,-5.0,16.0,18.0,19.0,20.0
Presentation,30.0,NaN,NaN,NaN,17.3,1.764594,14.0,16.0,18.0,18.75,20.0
Project,31.0,NaN,NaN,NaN,43.032258,4.408203,33.0,40.0,44.0,46.0,50.0
Enrollment_Date,31,19,2026-02-18,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Part A — Cleaning / 第一部分：数据清洗

In [5]:
clean = df.copy()
print('Raw shape / 原始形状:', df.shape)


Raw shape / 原始形状: (31, 14)


### 4.1 Clean column names / 清理列名

In [6]:
clean.columns = (
    clean.columns
         .str.strip()
         .str.lower()
         .str.replace(' ', '_', regex=False)
)
clean.columns.tolist()


['student_id',
 'name',
 'age',
 'gender',
 'major',
 'attendance_pct',
 'homework',
 'presentation',
 'project',
 'enrollment_date',
 'email',
 'city',
 'height_cm',
 'weight_kg']

### 4.2 Replace missing-value markers / 统一缺失值符号

In [7]:
missing_markers = ['', ' ', 'NA', 'N/A', 'na', 'n/a', '?', 'missing', 'Missing']
clean = clean.replace(missing_markers, np.nan)
clean.isna().sum().sort_values(ascending=False)


attendance_pct     1
gender             1
presentation       1
weight_kg          1
homework           1
name               0
major              0
age                0
student_id         0
project            0
enrollment_date    0
email              0
city               0
height_cm          0
dtype: int64

### 4.3 Trim text / 去除文本多余空格

In [8]:
text_cols = clean.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    clean[col] = clean[col].astype('string').str.strip()
clean[['name', 'gender', 'major', 'email', 'city']].head(8)


,name,gender,major,email,city
0,Ali Khan,Male,Computer Science,ali@example.com,Wuhan
1,Sara Li,F,computer science,sara.li@example.com,Wuhan
2,Zhang Wei,male,AI,zhang.wei@example.com,Huangshi
3,Fatima Noor,Female,Artificial Intelligence,fatima.noor@example.com,Huangshi
4,Chen Yu,M,CS,chen.yu@example.com,wuhan
5,Bilal Ahmad,MALE,Data Science,bilal@example.com,Wuhan
6,Liu Mei,female,data science,liu.mei@example.com,Huangshi
7,Ayesha Malik,F,AI,ayesha@example.com,Huangshi


### 4.4 Standardize gender / 统一性别类别

In [9]:
gender_map = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'}
clean['gender'] = clean['gender'].str.lower().map(gender_map)
clean['gender'].value_counts(dropna=False)


gender
Male      15
Female    15
NaN        1
Name: count, dtype: int64

### 4.5 Standardize major / 统一专业名称

In [10]:
major_key = clean['major'].str.lower().str.replace('.', '', regex=False)
major_map = {
    'cs': 'Computer Science',
    'computer science': 'Computer Science',
    'ai': 'Artificial Intelligence',
    'artificial intelligence': 'Artificial Intelligence',
    'data science': 'Data Science',
}
clean['major'] = major_key.map(major_map)
clean['major'].value_counts(dropna=False)


major
Computer Science           12
Artificial Intelligence    10
Data Science                9
Name: count, dtype: int64

### 4.6 Standardize city / 统一城市大小写

In [11]:
clean['city'] = clean['city'].str.title()
clean['city'].value_counts(dropna=False)


city
Wuhan       17
Huangshi    14
Name: count, dtype: Int64

### 4.7 Convert numeric columns / 转换数值列

In [12]:
numeric_cols = ['age', 'attendance_pct', 'homework', 'presentation',
                'project', 'height_cm', 'weight_kg']
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
clean[numeric_cols].dtypes


age                 Int64
attendance_pct    float64
homework          float64
presentation      float64
project             int64
height_cm           int64
weight_kg         float64
dtype: object

### 4.8 Validate ranges / 检查数值范围

In [13]:
rules = {
    'age': (16, 80),
    'attendance_pct': (0, 100),
    'homework': (0, 20),
    'presentation': (0, 20),
    'project': (0, 50),
    'height_cm': (130, 220),
    'weight_kg': (35, 200),
}
for col, (low, high) in rules.items():
    invalid = ~clean[col].between(low, high) & clean[col].notna()
    if invalid.any():
        print(f'{col}: invalid values / 无效值 ->', clean.loc[invalid, col].tolist())


age: invalid values / 无效值 -> [150, -3]
attendance_pct: invalid values / 无效值 -> [105.0]
homework: invalid values / 无效值 -> [-5.0]
height_cm: invalid values / 无效值 -> [250]


### 4.9 Replace impossible values with missing / 将不合理值设为缺失

In [14]:
for col, (low, high) in rules.items():
    clean.loc[~clean[col].between(low, high), col] = np.nan
clean[numeric_cols].isna().sum()


age               3
attendance_pct    2
homework          2
presentation      1
project           0
height_cm         1
weight_kg         1
dtype: int64

### 4.10 Parse dates / 解析日期

In [15]:
clean['enrollment_date'] = pd.to_datetime(
    clean['enrollment_date'], errors='coerce', format='mixed'
)
print('Invalid/missing dates / 无效或缺失日期:')
clean.loc[clean['enrollment_date'].isna(), ['student_id', 'name', 'enrollment_date']]


Invalid/missing dates / 无效或缺失日期:


,student_id,name,enrollment_date
20,1021,Lin Fang,NaT


### 4.11 Remove duplicates / 删除重复行

In [16]:
print('Duplicates before / 删除前重复行:', clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)
print('Shape after / 删除后形状:', clean.shape)


Duplicates before / 删除前重复行: 1
Shape after / 删除后形状: (30, 14)


### 4.12 Fill missing values with sklearn SimpleImputer / 用 sklearn 填充缺失值

In [17]:
# 数值列用中位数填充
num_imputer = SimpleImputer(strategy='median')
clean[numeric_cols] = num_imputer.fit_transform(clean[numeric_cols])

# 类别列用众数填充
for col in ['gender', 'major', 'city']:
    mode_value = clean[col].mode(dropna=True)[0]
    clean[col] = clean[col].fillna(mode_value)
    print(f'{col}: mode / 众数 = {mode_value}')

clean.isna().sum().sort_values(ascending=False)


gender: mode / 众数 = Male
major: mode / 众数 = Computer Science
city: mode / 众数 = Wuhan


enrollment_date    1
student_id         0
age                0
gender             0
major              0
name               0
attendance_pct     0
homework           0
presentation       0
project            0
email              0
city               0
height_cm          0
weight_kg          0
dtype: int64

### 4.13 Outlier detection (IQR / boxplot) / 异常值检测

In [18]:
def iqr_outlier_mask(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

mask, lower, upper = iqr_outlier_mask(clean['weight_kg'])
print(f'Weight IQR limits / 体重 IQR 范围: [{lower:.2f}, {upper:.2f}]')
if mask.any():
    print(f'Detected {mask.sum()} potential outliers / 检测到潜在异常值:')
    print(clean.loc[mask, ['student_id', 'name', 'weight_kg']])
else:
    print('No outliers detected / 未检测到异常值')


Weight IQR limits / 体重 IQR 范围: [27.50, 95.50]
No outliers detected / 未检测到异常值


### 4.14 Clean email / 清理邮箱

In [19]:
clean['email'] = clean['email'].str.strip().str.lower()
clean['email_valid'] = clean['email'].str.contains('@', na=False)
clean[['email', 'email_valid']].head(10)


,email,email_valid
0,ali@example.com,True
1,sara.li@example.com,True
2,zhang.wei@example.com,True
3,fatima.noor@example.com,True
4,chen.yu@example.com,True
5,bilal@example.com,True
6,liu.mei@example.com,True
7,ayesha@example.com,True
8,wang.jun@example.com,True
9,hassan@example.com,True


### 4.15 Final dtypes / 最终数据类型

In [20]:
clean['age'] = clean['age'].round().astype('Int64')
for col in ['gender', 'major', 'city']:
    clean[col] = clean[col].astype('category')
clean.dtypes


student_id                  int64
name                       string
age                         Int64
gender                   category
major                    category
attendance_pct            float64
homework                  float64
presentation              float64
project                   float64
enrollment_date    datetime64[us]
email                      string
city                     category
height_cm                 float64
weight_kg                 float64
email_valid               boolean
dtype: object

# Part B — Feature Engineering / 第二部分：特征工程

### 5.1 StandardScaler / 数值标准化

In [21]:
scaler = StandardScaler()
scale_cols = ['attendance_pct', 'homework', 'presentation',
              'project', 'height_cm', 'weight_kg']
scaled = pd.DataFrame(
    scaler.fit_transform(clean[scale_cols]),
    columns=[f'{c}_scaled' for c in scale_cols],
    index=clean.index
)
clean = pd.concat([clean, scaled], axis=1)
clean[[f'{c}_scaled' for c in scale_cols]].head()


,attendance_pct_scaled,homework_scaled,presentation_scaled,project_scaled,height_cm_scaled,weight_kg_scaled
0,0.982959,0.281788,-0.172917,0.015243,0.512079,0.273217
1,0.396118,0.810140,0.403473,0.472526,-0.636509,-0.995292
2,1.423089,1.338493,0.979864,1.158450,1.229947,0.761105
3,0.176052,-0.774917,-0.172917,-0.670682,-1.354377,-1.190447
4,-1.364405,-1.303269,-1.325698,-1.127965,1.660667,1.248994


### 5.2 One-hot encoding (get_dummies) / 独热编码

In [22]:
clean = pd.get_dummies(clean, columns=['city'], prefix='city', dtype=int)
clean.filter(like='city_').head()


,city_Huangshi,city_Wuhan
0,0,1
1,0,1
2,1,0
3,1,0
4,0,1


### 5.3 LabelEncoder / 标签编码

In [23]:
le_gender = LabelEncoder()
clean['gender_encoded'] = le_gender.fit_transform(clean['gender'])
print('gender mapping / 性别映射:', dict(zip(le_gender.classes_, range(len(le_gender.classes_)))))

le_major = LabelEncoder()
clean['major_encoded'] = le_major.fit_transform(clean['major'])
print('major mapping / 专业映射:', dict(zip(le_major.classes_, range(len(le_major.classes_)))))


gender mapping / 性别映射: {'Female': 0, 'Male': 1}
major mapping / 专业映射: {'Artificial Intelligence': 0, 'Computer Science': 1, 'Data Science': 2}


# Part C — Validation / 第三部分：清洗后验证

In [24]:
print('Final shape / 最终形状:', clean.shape)
print('Duplicates / 重复行:', clean.duplicated().sum())
print('Missing cells / 缺失单元格:', int(clean.isna().sum().sum()))


Final shape / 最终形状: (30, 24)
Duplicates / 重复行: 0
Missing cells / 缺失单元格: 1


In [25]:
checks = {
    'age_valid': clean['age'].between(16, 80).all(),
    'attendance_valid': clean['attendance_pct'].between(0, 100).all(),
    'homework_valid': clean['homework'].between(0, 20).all(),
    'presentation_valid': clean['presentation'].between(0, 20).all(),
    'project_valid': clean['project'].between(0, 50).all(),
    'height_valid': clean['height_cm'].between(130, 220).all(),
    'weight_valid': clean['weight_kg'].between(35, 200).all(),
}
checks


{'age_valid': np.True_,
 'attendance_valid': np.True_,
 'homework_valid': np.True_,
 'presentation_valid': np.True_,
 'project_valid': np.True_,
 'height_valid': np.True_,
 'weight_valid': np.True_}

In [26]:
clean.head(10)

,student_id,name,age,gender,major,attendance_pct,homework,presentation,project,enrollment_date,email,height_cm,weight_kg,email_valid,attendance_pct_scaled,homework_scaled,presentation_scaled,project_scaled,height_cm_scaled,weight_kg_scaled,city_Huangshi,city_Wuhan,gender_encoded,major_encoded
0,1001,Ali Khan,20,Male,Computer Science,92.0,18.0,17.0,43.0,2026-02-15,ali@example.com,173.0,65.0,True,0.982959,0.281788,-0.172917,0.015243,0.512079,0.273217,0,1,1,1
1,1002,Sara Li,19,Female,Computer Science,88.0,19.0,18.0,45.0,2026-02-15,sara.li@example.com,165.0,52.0,True,0.396118,0.810140,0.403473,0.472526,-0.636509,-0.995292,0,1,0,1
2,1003,Zhang Wei,21,Male,Artificial Intelligence,95.0,20.0,19.0,48.0,2026-02-16,zhang.wei@example.com,178.0,70.0,True,1.423089,1.338493,0.979864,1.158450,1.229947,0.761105,1,0,1,0
3,1004,Fatima Noor,20,Female,Artificial Intelligence,86.5,16.0,17.0,40.0,2026-02-17,fatima.noor@example.com,160.0,50.0,True,0.176052,-0.774917,-0.172917,-0.670682,-1.354377,-1.190447,1,0,0,0
4,1005,Chen Yu,20,Male,Computer Science,76.0,15.0,15.0,38.0,2026-02-18,chen.yu@example.com,181.0,75.0,True,-1.364405,-1.303269,-1.325698,-1.127965,1.660667,1.248994,0,1,1,1
5,1006,Bilal Ahmad,22,Male,Data Science,86.5,18.0,16.0,44.0,2026-02-18,bilal@example.com,176.0,72.0,True,0.176052,0.281788,-0.749308,0.243884,0.942800,0.956261,0,1,1,2
6,1007,Liu Mei,19,Female,Data Science,83.0,18.0,18.0,42.0,2026-02-18,liu.mei@example.com,158.0,47.0,True,-0.337434,0.281788,0.403473,-0.213399,-1.641524,-1.483180,1,0,0,2
7,1008,Ayesha Malik,20,Female,Artificial Intelligence,91.0,20.0,20.0,49.0,2026-02-19,ayesha@example.com,164.0,55.0,True,0.836248,1.338493,1.556254,1.387092,-0.780083,-0.702559,1,0,0,0
8,1009,Wang Jun,20,Male,Computer Science,72.0,14.0,15.0,35.0,2026-02-19,wang.jun@example.com,175.0,68.0,True,-1.951246,-1.831622,-1.325698,-1.813890,0.799226,0.565950,0,1,1,1
9,1010,Hassan Raza,21,Male,Data Science,68.0,13.0,14.0,33.0,2026-02-20,hassan@example.com,172.0,80.0,True,-2.538087,-2.359974,-1.902088,-2.271173,0.368505,1.736882,1,0,1,2


# Part D — Save / 第四部分：保存清洗结果

In [27]:
OUTPUT_CSV = Path('/workspace/student_data_cleaned.csv')
OUTPUT_EXCEL = Path('/workspace/student_data_cleaned.xlsx')

clean.to_csv(OUTPUT_CSV, index=False)
clean.to_excel(OUTPUT_EXCEL, index=False)

print('Saved / 已保存:', OUTPUT_CSV.resolve())
print('Saved / 已保存:', OUTPUT_EXCEL.resolve())


Saved / 已保存: /workspace/student_data_cleaned.csv
Saved / 已保存: /workspace/student_data_cleaned.xlsx
